In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/cosesmic/layers/rasters

 landslide_sample.gpkg		  utm_LULC.tif
 max_rainfall.tif		  utm_NDVI_districts.tif
 mean_rainfall.tif		  utm_ndvi.tif
 monsoon_rainfall.tif		 'utm_Plan Curvature.tif'
 OUTPUT.vrt			 'utm_Profile Curvature.tif'
 proximity_fault_clipped.tif	  utm_sand_0_40.tif
 proximity_faults.tif		  utm_silt_0_40.tif
 proximity_river.tif		  utm_Slope.tif
 proximity_roads.tif		  utm_soc_0_40.tif
 utm_Aspect_class.tif		  utm_soil_type.tif
 utm_clay_0_40.tif		 'utm_Terrain Ruggedness Index.tif'
 utm_Elevation.tif		 'utm_Terrain Ruggedness Index (TRI).tif'
'utm_Flow Accumulation.tif'	 'utm_Topographic Wetness Index.tif'
'utm_log_Flow Accumulation.tif'   utm_Uttarakhand_Rock_Glacier_Mask.tif
 utm_LULC_districts.tif


In [ ]:
import rasterio, numpy as np

with rasterio.open("/content/drive/MyDrive/cosesmic/layers/rasters/utm_Slope.tif") as src:
    data = src.read(1)
    print("Min:", np.nanmin(data))
    print("Max:", np.nanmax(data))

Min: -99999.0
Max: 1.5646517


In [ ]:
input_folder = "/content/drive/MyDrive/cosesmic/layers/raster_class/"
output_folder = "/content/drive/MyDrive/cosesmic/layers/output_weight_rasters"
fr_csv_path = "/content/drive/MyDrive/thesis/processed/FR_results_features_masked.csv"

slope_path ="/content/drive/MyDrive/cosesmic/layers/rasters/utm_Slope.tif" # RAW slope raster

os.makedirs(output_folder, exist_ok=True)

In [ ]:
raster_to_factor = {
    "slope": "slope",
    "utm_Aspect_class": "utm_Aspect",
    "elevation": "elevation",
    "plan_curvature": "plan_curv",
    "profile_curvature": "profile_curv",
    "twi": "twi",
    "tri": "tri",
    "ndvi": "ndvi",
    "lulc": "utm_lulc",
    "proximity_roads": "proximity_roads",
    "proximity_rivers": "proximity_rivers",
    "proximity_fault": "proximity_fault",
    "mean_rainfall_class": "mean_rainfall",
    "max_rainfall_class": "max_rainfall",
    "mean_monsoon_class": "monsoon_rainfall",
    "log_flow_accumulation": "flow_accumulation",
}

In [ ]:
fr_df = pd.read_csv(fr_csv_path)

# Lookup: {factor: {class: weight}}
fr_lookup = {}
for factor, group in fr_df.groupby("factor"):
    fr_lookup[factor] = dict(zip(group["class"].astype(float), group["weight"]))

In [ ]:
# adjust based on your slope units
threshold = 0.2  # radians (~11.5°)
# if degrees → threshold = 10

In [ ]:
outputs = []

with rasterio.open(slope_path) as slope_src:

    for raster_name, factor in raster_to_factor.items():

        input_path = os.path.join(input_folder, raster_name + ".tif")
        output_path = os.path.join(output_folder, f"W_{raster_name}.tif")

        if not os.path.exists(input_path):
            print(f"Missing: {input_path}")
            continue

        if factor not in fr_lookup:
            print(f"Factor not in FR table: {factor}")
            continue

        class_to_weight = fr_lookup[factor]

        try:
            with rasterio.open(input_path) as src:

                profile = src.profile.copy()
                nodata = src.nodata if src.nodata is not None else -9999

                profile.update(
                    dtype=rasterio.float32,
                    nodata=-9999,
                    compress="LZW",
                    tiled=True,
                    blockxsize=256,
                    blockysize=256
                )

                with rasterio.open(output_path, "w", **profile) as dst:

                    for _, window in src.block_windows(1):

                        data = src.read(1, window=window).astype(float)
                        slope = slope_src.read(1, window=window).astype(float)

                        out = np.full_like(data, -9999, dtype=np.float32)

                        # valid mask
                        valid = (
                            (data != nodata) &
                            (slope >= threshold)
                        )

                        if np.any(valid):

                            classes = np.unique(data[valid])

                            for cls in classes:
                                weight = class_to_weight.get(float(cls), 0.0)

                                out[(data == cls) & valid] = weight

                        dst.write(out, 1, window=window)

                outputs.append(output_path)
                print(f"Done: {raster_name}")

        except Exception as e:
            print(f"Error: {raster_name} → {e}")

print(f"\nGenerated {len(outputs)} weight rasters")

Done: slope
Done: utm_Aspect_class
Done: elevation
Done: plan_curvature
Done: profile_curvature
Done: twi
Done: tri
Done: ndvi
Done: lulc
Done: proximity_roads
Done: proximity_rivers
Done: proximity_fault
Done: mean_rainfall_class
Done: max_rainfall_class
Done: mean_monsoon_class
Done: log_flow_accumulation

Generated 16 weight rasters
